# GeoNUSAF — Fig. 11 only: synthetic pairs, painted on demand

One figure. You give it **generated masks**; it paints the matching images with the
Stage-2 generator, reads them back with the frozen scorer, and draws the 3 x 3 grid.

```
fig11_synthetic_pairs.pdf   190 x 152 mm   figure*
   Fig. 11  ->  Section 6.4, at the end of the subsection, after the presence-rate sentence
```

| column | where it comes from |
|---|---|
| generated mask | your `genmask{i}.png` — colour or class-index, both decode |
| generated image | painted here by `Pranilllllll/geonusaf-stage2-mask2image-block-fold1-D` |
| frozen scorer read-back | `Pranilllllll/segformer-b0-kaggle-ktm-dataset-block-fold1` on that image |

Nothing about the figure's size, fonts, palette or caption has changed from the
six-figure notebook. Only the middle column's source has: it used to look for a
`genimage{i}.png` on disk and now makes one.

**Before you run.** Internet ON (three Hub repos are pulled, ~2.5 GB the first time),
GPU on (T4 is what Stage 3 used; ~9 s per image), and the folder holding your masks
attached. CPU works but budget several minutes per image.

**Reproducing the published pair.** Stage 3 keyed each latent to the absolute pair
index: `torch.Generator().manual_seed(42 + 777 + 1000*R + i)`, one draw per image.
With `SEED_MODE = "stage3"` and `R_SET` matching the set your masks came from, index
36 repaints the same image that sits in `fake_pairs_R1/genimage36.png` — the one that
actually went into Stage-4 training. Set `SEED_MODE = "fixed"` if the masks are from
somewhere else and the index carries no such meaning.


## 0 · Control panel

In [ ]:
# =====================================================================================
#   CONTROL PANEL  -  the only cell you normally touch.  Set it, then Run All.
# =====================================================================================

# --------------------------------------------------------------------------- Fig. 11
# One row per panel row. A slot takes a stem ("genmask36"), a file name
# ("genmask36.png") or a full path - all three resolve.
FIG11_GENMASKS  = ["genmask36.png", "genmask50.png", "genmask53.png"]
FIG11_LABELS    = ["high fidelity", "mid fidelity", "low fidelity, river-bearing"]

# Leave these empty and the image is painted from the mask above. Fill one in (a path
# to an existing genimage) and that file is used for that row instead - useful if you
# already hold the exact pair and only want the read-back computed.
FIG11_GENIMAGES = ["", "", ""]

# Indices only matter for the seed (see SEED_MODE). "" = read the index off the mask
# file name, which is what you want when the names are genmask{i}.png.
FIG11_INDICES   = ["", "", ""]

# ---- where the masks are. "auto" scans /kaggle/input for a folder of genmask*.png.
GENMASK_DIR = "auto"

# ---- painting ---------------------------------------------------------------------
PAINT      = "auto"        # "auto" = paint missing images · "off" = never paint
REPAINT    = False         # True ignores the cache in figs/_painted and paints again
R_SET      = 1             # which fake_pairs_R set these masks belong to (1, 2 or 3)
SEED_MODE  = "stage3"      # "stage3" = reproduce that set's published pair · "fixed"
FIXED_SEED = 0             # only used when SEED_MODE = "fixed"

# Sampling. None = inherit from the Stage-2 checkpoint, which is what Stage 3 did
# (DDIM 30, guidance 3.5, cond scale 1.0, canny on at 0.4). Override only to explore.
DDIM_STEPS  = None
GUIDANCE    = None
COND_SCALE  = None
CANNY_SCALE = None
USE_CANNY   = None

# ---- repos ------------------------------------------------------------------------
S2_REPO = "Pranilllllll/geonusaf-stage2-mask2image-block-fold1-D"   # the painter
HF_B0   = "Pranilllllll/segformer-b0-kaggle-ktm-dataset-block-fold1"  # the scorer
INFER   = "auto"           # "off" skips every download; panels say what is missing

# ---- house style ------------------------------------------------------------------
USE_DISPLAY_PALETTE = True    # False renders the released mask colours instead
S3_SEED = 42                  # Stage 3's cfg.SEED - part of the latent key
IMAGE_SEED_BASE = 777         # Stage 3's cfg.IMAGE_SEED_BASE - the other part

FIG11_ROWS = [dict(label=(FIG11_LABELS[r] if r < len(FIG11_LABELS) else ""),
                   mask=(FIG11_GENMASKS[r] if r < len(FIG11_GENMASKS) else ""),
                   image=(FIG11_GENIMAGES[r] if r < len(FIG11_GENIMAGES) else ""),
                   index=(FIG11_INDICES[r] if r < len(FIG11_INDICES) else ""))
              for r in range(3)]

print("control panel set ·", sum(1 for r in FIG11_ROWS if r["mask"]), "mask slot(s) filled",
      "· seed mode", SEED_MODE, "· R%d" % R_SET)

## 1 · Environment

`diffusers` for the SD 1.5 UNet, the VAE and both ControlNets, `peft` for the LoRA
adapter, `transformers` for CLIP and for the frozen scorer — the same four the
Stage-2 and Stage-3 notebooks instantiate, because this cell rebuilds the same objects.

`transformers` 5.x renamed `decode_head.linear_c`, which changes every key in the
scorer's checkpoint. The cell reports the version rather than silently loading a model
with half its weights missing.

In [ ]:
!pip install -q "diffusers>=0.27.0" "peft>=0.10.0" 2>&1 | tail -2

import os, sys, json, math, time, re, shutil
from pathlib import Path

HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass                      # all three repos are public; a token only raises rate limits

TORCH_OK = True
try:
    import torch
    print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("  device:", torch.cuda.get_device_name(0))
    else:
        print("  NO GPU. Painting will work but takes minutes per image, not seconds.")
except Exception as e:
    TORCH_OK = False
    print("torch not importable:", e)

try:
    import transformers
    print("transformers", transformers.__version__)
    if int(transformers.__version__.split(".")[0]) >= 5:
        print("  WARNING: 5.x renames decode_head.linear_c -> linear_projections, so the")
        print("  scorer's checkpoint keys will not match. Run:")
        print("      !pip install -q 'transformers<5'   then restart the kernel.")
except Exception as e:
    print("transformers not importable:", e)

## 2 · Setup, fonts and house style

Identical to the six-figure notebook: Times/serif stack, 7 pt floor, Type 42 fonts,
constrained layout, sub-labels placed after layout is resolved. Copied rather than
rewritten, so this figure cannot drift from the other twenty-one.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, TwoSlopeNorm
from matplotlib.patches import Patch, FancyBboxPatch, FancyArrowPatch, Circle, Rectangle
from matplotlib.lines import Line2D
import matplotlib.font_manager as fm

OUT = Path("figs"); OUT.mkdir(exist_ok=True)
SAVE_EPS = False     # Elsevier accepts EPS or PDF; PDF is the safer of the two
SAVE_PNG = True      # 600 dpi previews for checking. Submit the PDFs, not these.

MM = 1 / 25.4
W1, W15, W2 = 90 * MM, 140 * MM, 190 * MM     # Elsevier: single / 1.5 / double column

# ---- Times. Liberation Serif and Nimbus Roman are metrically identical to Times New Roman,
# ---- so on Kaggle or Linux the output is indistinguishable in print.
SERIF = ["Times New Roman", "Nimbus Roman", "Liberation Serif", "TeX Gyre Termes",
         "FreeSerif", "DejaVu Serif"]
have = {f.name for f in fm.fontManager.ttflist}
resolved = next((f for f in SERIF if f in have), "DejaVu Serif")
print("serif family in use:", resolved)
if resolved == "DejaVu Serif":
    print("  Times not found. On Kaggle run:  !apt-get -qq install -y fonts-liberation")
    print("  then restart the kernel. DejaVu Serif is a safe stand-in meanwhile.")

mpl.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 600,
    "savefig.facecolor": "white",
    "savefig.transparent": False,
    "pdf.fonttype": 42,          # embed TrueType. Type 3 fails Elsevier preflight.
    "ps.fonttype": 42,
    "font.family": "serif",
    "font.serif": SERIF,
    "mathtext.fontset": "stix",  # Times-compatible math
    "font.size": 7,              # Elsevier floor is 7 pt at final size
    "axes.labelsize": 7.5,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
    "legend.fontsize": 7,
    "legend.frameon": False,
    "legend.handlelength": 1.3,
    "legend.handletextpad": 0.5,
    "legend.columnspacing": 1.4,
    "legend.borderpad": 0.0,
    "axes.titlesize": 7.5,
    "axes.linewidth": 0.6,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "grid.linewidth": 0.4,
    "grid.color": "0.88",
    "lines.linewidth": 1.0,
    "lines.markersize": 3.5,
    "xtick.major.width": 0.6,
    "ytick.major.width": 0.6,
    "xtick.major.size": 2.4,
    "ytick.major.size": 2.4,
    "hatch.linewidth": 0.5,
})

SUBLABEL_PT = 8      # size of (a) (b) (c)
# Panels stacked one per row share a single x, so (a) and (b) sit on one vertical
# line: a wide image and a chart carrying long tick labels do not have the same
# centre, and centring each label on its own panel puts them at two different x.
#   "union" (default) · "mean" · a figure fraction such as 0.5 · None = per panel
SUBLABEL_STACK_X = "union"
NOTE = dict(fontsize=5.6, color="0.55", style="italic")   # rare in-figure note style

REGISTER = []

def top_legend(fig, handles, labels, ncol, loc="outside upper center"):
    # Legend above the plot, outside the axes, so it can never sit on the data.
    return fig.legend(handles=handles, labels=labels, loc=loc, ncol=ncol,
                      frameon=False, fontsize=7, borderaxespad=0.15)

def finish(fig, name, figure_no, section, where, float_env, caption, axes=None, hspace=0.12):
    # axes: pass the panel axes in order to get (a) (b) (c) below them, on one baseline.
    # An entry may itself be a list of axes that share one sub-label (Fig. 1b).
    if axes:
        h_in = fig.get_size_inches()[1]
        reserve = (5.6 * MM) / h_in            # strip of figure height kept free for the labels
        eng = fig.get_layout_engine()

        from matplotlib.transforms import Bbox

        def boxes_now():
            fig.draw_without_rendering()
            r = fig.canvas.get_renderer()
            out = []
            for item in axes:
                group_ = item if isinstance(item, (list, tuple)) else [item]
                bs = [a.get_tightbbox(r).transformed(fig.transFigure.inverted())
                      for a in group_]
                out.append(Bbox.from_extents(min(b.x0 for b in bs), min(b.y0 for b in bs),
                                             max(b.x1 for b in bs), max(b.y1 for b in bs)))
            return out

        eng.set(rect=(0, reserve, 1, 1 - reserve))   # (left, bottom, width, height)
        bx = boxes_now()

        def cluster(bs):
            # group panels into rows by vertical centre, top row first
            idx = sorted(range(len(bs)), key=lambda i: -0.5 * (bs[i].y0 + bs[i].y1))
            rows, cur = [], [idx[0]]
            for i in idx[1:]:
                c0 = 0.5 * (bs[cur[-1]].y0 + bs[cur[-1]].y1)
                c1 = 0.5 * (bs[i].y0 + bs[i].y1)
                if abs(c0 - c1) < 0.06:
                    cur.append(i)
                else:
                    rows.append(cur); cur = [i]
            rows.append(cur)
            return rows

        rows = cluster(bx)
        if len(rows) > 1:            # stacked panels need a gap to hold the label
            eng.set(hspace=hspace)
            bx = boxes_now(); rows = cluster(bx)
        stack = len(rows) > 1 and all(len(r) == 1 for r in rows)
        xcom = None
        if stack and SUBLABEL_STACK_X is not None:
            if isinstance(SUBLABEL_STACK_X, (int, float)):
                xcom = float(SUBLABEL_STACK_X)
            elif SUBLABEL_STACK_X == "mean":
                xcom = float(np.mean([0.5 * (bx[r[0]].x0 + bx[r[0]].x1) for r in rows]))
            else:
                xcom = 0.5 * (min(b.x0 for b in bx) + max(b.x1 for b in bx))
        for row in rows:
            ybase = min(bx[i].y0 for i in row) - 0.30 * reserve
            for i in row:
                fig.text(xcom if xcom is not None else 0.5 * (bx[i].x0 + bx[i].x1),
                         ybase, "(" + "abcdefgh"[i] + ")",
                         ha="center", va="top", fontsize=SUBLABEL_PT)
        fig.set_layout_engine("none")

    fig.savefig(OUT / (name + ".pdf"))
    if SAVE_EPS:
        fig.savefig(OUT / (name + ".eps"))
    if SAVE_PNG:
        fig.savefig(OUT / (name + ".png"))
    w, h = fig.get_size_inches()
    size = str(round(w / MM)) + " x " + str(round(h / MM)) + " mm"
    print(name + ".pdf   " + size + "   " + float_env)
    print("   " + figure_no + "  ->  " + section + ", " + where)
    REGISTER.append(dict(file=name + ".pdf", figure=figure_no, section=section,
                         insertion_point=where, float_env=float_env, size=size,
                         caption=caption))

print("ready. output ->", OUT.resolve())

## 3 · Palette, decoding and slots

The class order and the released mask colours are the ones in your training notebook's
`BGR_TO_CLASS`. Masks are decoded by measuring the file, so the colour masks you
extracted, byte-reversed masks and the index-coded `{0..5} + 255` contract masks all
work without a switch.

`to_index` is also what feeds the generator: whatever the mask looks like on disk, the
control branch receives the same 7-channel one-hot it was trained on.

In [ ]:
import numpy as np
from PIL import Image

RASTER = (".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp")
CACHE = OUT / "_cache"; CACHE.mkdir(parents=True, exist_ok=True)
PAINTED = OUT / "_painted"; PAINTED.mkdir(parents=True, exist_ok=True)

# ---- classes in the order the checkpoints use (model cards, and BGR_TO_CLASS in the
# ---- training notebook). Everything in this notebook indexes classes this way.
CLASSES = ["Residential", "Road", "River", "Forest", "UnusedLand", "Agricultural"]
N_CODES = 7                 # 0..5 semantic + 6 = ignore, inside both generators
IGNORE_INDEX, IGNORE_CODE = 255, 6
IMG_SIZE, LATENT_SIZE = 512, 64

# The released masks are written with cv2, so the file's RGB values are these:
RELEASED = {"Residential": "#000080", "Road": "#008000", "River": "#800000",
            "Forest": "#808000", "UnusedLand": "#008080", "Agricultural": "#800080",
            "Unannotated": "#000000"}
DISPLAY = {"Residential": "#C0392B", "Road": "#4D4D4D", "River": "#1F6FB4",
           "Forest": "#14532D", "UnusedLand": "#C2A477", "Agricultural": "#E8B62C",
           "Unannotated": "#FFFFFF"}
CLR = DISPLAY if USE_DISPLAY_PALETTE else RELEASED
print("palette:", "display" if USE_DISPLAY_PALETTE else "released")


def hex2rgb(h):
    return np.array([int(h[i:i + 2], 16) for i in (1, 3, 5)], np.uint8)


def _key(rgb):
    return (int(rgb[0]) << 16) | (int(rgb[1]) << 8) | int(rgb[2])


_DIGIT_RUN = re.compile(r"\d+")


def listdir(d):
    if not d or not Path(str(d)).is_dir():
        return []
    return sorted(str(p) for p in Path(d).iterdir() if p.suffix.lower() in RASTER)


def stem(p):
    return Path(p).stem


def split_stem(s):
    """genmask36 -> ("genmask", 36, ""). The LAST run of digits is the index."""
    s = str(s)
    runs = list(_DIGIT_RUN.finditer(s))
    if not runs:
        return s.lower(), None, ""
    g = runs[-1]
    return s[:g.start()].lower(), int(g.group()), s[g.end():].lower()


def _num(s):
    return split_stem(s)[1]


def tile_key(s):
    pre, n, suf = split_stem(s)
    return None if n is None else (n, suf)


def load_rgb(path, bgr=False):
    """Read an image as RGB. bgr=True for files written with cv2.imwrite."""
    try:
        a = np.array(Image.open(path).convert("RGB"))
    except Exception:
        return None
    return a[..., ::-1].copy() if bgr else a


_COLOUR_LUTS = []
for _pal in (RELEASED, DISPLAY):
    for _rev in (False, True):
        _COLOUR_LUTS.append({_key(hex2rgb(v)[::-1] if _rev else hex2rgb(v)):
                             (CLASSES.index(k) if k in CLASSES else 255)
                             for k, v in _pal.items()})


def to_index(mask):
    """Any mask -> HxW class indices (255 = unannotated). Measured, not assumed."""
    if mask is None:
        return None, 0.0
    a = mask if mask.ndim == 3 else np.dstack([mask] * 3)
    grey = np.array_equal(a[..., 0], a[..., 1]) and np.array_equal(a[..., 1], a[..., 2])
    if grey and len(np.unique(a[..., 0])) <= 8:                 # index-coded {0..5} + 255
        g = a[..., 0].astype(np.int16)
        out = np.where(g < len(CLASSES), g, 255).astype(np.int16)
        return out, float((out != 255).mean())
    keys = (a[..., 0].astype(np.int32) << 16) | (a[..., 1].astype(np.int32) << 8) | a[..., 2]
    best, cov = None, -1.0
    for lut in _COLOUR_LUTS:
        out = np.full(keys.shape, 255, np.int16)
        for k, ci in lut.items():
            out[keys == k] = ci
        c = float((out != 255).mean())
        if c > cov:
            best, cov = out, c
    return best, cov


def paint(idx):
    """class indices -> the figure palette."""
    out = np.zeros(idx.shape + (3,), np.uint8)
    out[:] = hex2rgb(CLR["Unannotated"])
    for k, c in enumerate(CLASSES):
        out[idx == k] = hex2rgb(CLR[c])
    return out


def to_512(idx):
    """Class indices at any size -> uint8 {0..5, 255} at 512, nearest neighbour.

    The Stage-2 control branch downsamples its conditioning by 8, so a 512 mask gives
    the 64x64 condition the latent expects. Anything else fails deep inside ControlNet
    with "tensor a (64) must match tensor b (16)".
    """
    a = np.where(idx < 0, 255, idx)
    a = np.where(a > 5, 255, a).astype(np.uint8)
    if a.shape != (IMG_SIZE, IMG_SIZE):
        a = np.array(Image.fromarray(a).resize((IMG_SIZE, IMG_SIZE), Image.NEAREST))
    return a


def find_file(spec, pool):
    """A stem, a file name, or a path -> a path from pool (or None)."""
    if not spec:
        return None
    spec = str(spec)
    if Path(spec).is_file():
        return spec
    if Path(spec).suffix.lower() in RASTER or "/" in spec or "\\" in spec:
        if str(Path(spec).parent) not in (".", ""):
            print("   NOTE: %s is not on disk; matching on the name %s."
                  % (spec, Path(spec).name))
        spec = Path(spec).stem
    for p in pool:                                     # 1. the same file stem
        if stem(p) == spec:
            return p
    low = spec.lower()
    for p in pool:                                     # 2. the same stem, any case
        if stem(p).lower() == low:
            return p
    k = tile_key(spec)
    if k is None:
        return None
    hits = [p for p in pool if tile_key(stem(p)) == k]  # 3. index + suffix
    if len(hits) == 1:
        return hits[0]
    if len(hits) > 1:
        hits.sort(key=lambda p: (len(stem(p)), stem(p)))
        print("   NOTE: %d files are index %d; using %s." % (len(hits), k[0], stem(hits[0])))
        return hits[0]
    hits = [p for p in pool if _num(stem(p)) == k[0]]   # 4. index alone, last resort
    if len(hits) == 1:
        return hits[0]
    if hits:
        print("   NOTE: %d files carry index %d (%s); using the first."
              % (len(hits), k[0], ", ".join(stem(h) for h in hits[:4])))
        return hits[0]
    return None


def show(ax, arr, note="", frame="0.6"):
    ax.set_xticks([]); ax.set_yticks([])
    for sp in ax.spines.values():
        sp.set_linewidth(0.5); sp.set_color(frame)
    if arr is None:
        ax.add_patch(Rectangle((0, 0), 1, 1, transform=ax.transAxes, fc="#F4F4F4",
                               ec="#B2182B", lw=0.8))
        if note:
            ax.text(0.5, 0.5, note, transform=ax.transAxes, ha="center", va="center",
                    fontsize=6.2, color="#B2182B")
        return
    ax.imshow(arr, interpolation="nearest")


# the legend keeps the reading order used by the other 21 figures, so the whole
# figure set looks like one set; the colours per class are identical either way
LEGEND_ORDER = ["Residential", "Agricultural", "UnusedLand", "Forest", "Road", "River"]


def class_legend(fig, ncol=7):
    return top_legend(fig, [Patch(fc=CLR[c], ec="0.4", lw=0.3) for c in LEGEND_ORDER]
                      + [Patch(fc=CLR["Unannotated"], ec="0.4", lw=0.3)],
                      LEGEND_ORDER + ["unannotated"], ncol=ncol)


print("io ready")

## 4 · Finding the masks

`GENMASK_DIR = "auto"` walks the attached inputs for a folder holding `genmask*.png`.
It reports what it found and what each of your three slots resolved to, because a slot
that silently resolves to nothing is the one failure mode that produces a figure you
cannot tell is wrong.

In [ ]:
def _find_genmask_dirs(roots=("/kaggle/input", "input", "data", ".")):
    hits = {}
    for root in roots:
        root = Path(root)
        if not root.exists():
            continue
        for p in root.rglob("*"):
            try:
                if not p.is_dir() or str(OUT.resolve()) in str(p.resolve()):
                    continue
                n = sum(1 for q in p.iterdir()
                        if q.suffix.lower() in RASTER and q.name.lower().startswith("genmask"))
                if n:
                    hits[str(p)] = n
            except Exception:
                continue
    return hits


if GENMASK_DIR == "auto":
    _hits = _find_genmask_dirs()
    for d, n in sorted(_hits.items(), key=lambda kv: -kv[1])[:6]:
        print("   %5d genmask*.png  <-  %s" % (n, d))
    GENMASK_DIR = max(_hits.items(), key=lambda kv: kv[1])[0] if _hits else ""

GEN_MASKS = [p for p in listdir(GENMASK_DIR) if Path(p).name.lower().startswith("genmask")]
if not GEN_MASKS:                       # a folder of masks under some other naming
    GEN_MASKS = listdir(GENMASK_DIR)
GEN_IMGS = [p for p in listdir(GENMASK_DIR) if Path(p).name.lower().startswith("genimage")]

print("\nmasks : %d  <- %s" % (len(GEN_MASKS), GENMASK_DIR or "NOT FOUND"))
print("paired images already on disk: %d" % len(GEN_IMGS))
if not GEN_MASKS:
    print("\nNothing found. Attach the folder, or set GENMASK_DIR to it by hand.")

print("\nslots")
for _r, _row in enumerate(FIG11_ROWS):
    _p = find_file(_row["mask"], GEN_MASKS)
    _i = _row["index"] or (_num(stem(_p)) if _p else None)
    print("   row %d  %-18s -> %s   index %s"
          % (_r, _row["mask"] or "(empty)", Path(_p).name if _p else "NOT RESOLVED", _i))
    if _p is not None:
        _m, _cov = to_index(load_rgb(_p))
        _sh = {CLASSES[k]: float((_m == k).mean()) for k in range(len(CLASSES))}
        print("          decoded %.1f%% of pixels · %s" % (
            100 * _cov, ", ".join("%s %.1f%%" % (c[:4], 100 * v)
                                  for c, v in _sh.items() if v > 0.002)))

## 5 · Stage 2 — the painter, loaded frozen from the Hub

Rebuilt exactly as Stage 3's section 12 built it, and loaded from the same `best.pt`:

| piece | state | source |
|---|---|---|
| SD 1.5 UNet | frozen, fp16 | `S2_CFG["SD_MODEL_ID"]` |
| LoRA r16 a16 on `to_q/to_k/to_v/to_out.0` | trained | `best.pt["lora_state"]` |
| seg control branch, `conv_in` widened 3 -> 7 | trained | `best.pt["control_state"]` |
| canny control branch | pretrained, off-the-shelf | `S2_CFG["CANNY_ID"]` at scale 0.4 |
| VAE, CLIP text encoder, tokenizer | frozen | `S2_CFG["SD_MODEL_ID"]` |

Three things here are copied verbatim rather than re-derived, because getting any of
them subtly wrong produces a plausible image that is not the one the paper's pipeline
makes: the **condition** is a 7-channel one-hot of `{0..5} + ignore`, never palette RGB;
the **canny** channel is class-boundary edges of the same mask, tripled, added as a
second control branch at 0.4; and the **caption** is composed from the mask's own class
shares by `caption_from_index`, with an empty negative prompt.

Sampling hyper-parameters are inherited from the checkpoint unless you overrode them
in the control panel — the cell prints which of the two each one came from.

In [ ]:
PAINT_STATUS = "not attempted"
ENG = None

_MEAN = np.array([0.485, 0.456, 0.406], np.float32)
_STD = np.array([0.229, 0.224, 0.225], np.float32)

NOUN = {0: "residential rooftops", 1: "road network", 2: "river",
        3: "forest", 4: "bare unused land", 5: "agricultural fields"}
AREA_CLASSES = {0, 3, 4, 5}
THIN_CLASSES = (1, 2)
THIN_FLOOR = 0.005
EYE7 = np.eye(N_CODES, dtype=np.float32)


def to_code(lab):
    return np.where(lab == IGNORE_INDEX, IGNORE_CODE, lab).astype(np.uint8)


def cond_from_index(lab):
    return np.ascontiguousarray(EYE7[to_code(lab)].transpose(2, 0, 1))


def canny_from_index(lab):
    code = to_code(lab)
    e = np.zeros(code.shape, np.uint8)
    dh = (code[:, 1:] != code[:, :-1]); dv = (code[1:, :] != code[:-1, :])
    e[:, 1:] |= dh; e[:, :-1] |= dh; e[1:, :] |= dv; e[:-1, :] |= dv
    return np.ascontiguousarray(np.repeat(e[None].astype(np.float32), 3, axis=0))


def caption_from_index(lab):
    code = to_code(lab)
    h = np.bincount(code.ravel(), minlength=N_CODES)[:len(CLASSES)].astype(np.float64)
    s = h / max(h.sum(), 1.0)
    keep = [int(c) for c in np.argsort(-s) if s[c] >= 0.02][:3]
    for c in THIN_CLASSES:
        if s[c] >= THIN_FLOOR and c not in keep:
            keep.append(c)
    parts = []
    for c in sorted(keep, key=lambda c: -s[c]):
        q = ""
        if c in AREA_CLASSES:
            q = "dense " if s[c] >= 0.35 else ("scattered " if s[c] < 0.10 else "")
        parts.append(q + NOUN[c])
    return "aerial satellite view of Kathmandu Valley, " + ", ".join(parts)


def build_painter():
    """SD 1.5 + LoRA + the trained control branch, exactly as Stage 3 section 12."""
    import torch
    import torch.nn as nn
    from diffusers import (AutoencoderKL, UNet2DConditionModel, ControlNetModel,
                           DDIMScheduler)
    from transformers import CLIPTextModel, CLIPTokenizer
    from peft import LoraConfig
    from huggingface_hub import hf_hub_download

    dev = "cuda" if torch.cuda.is_available() else "cpu"
    half = torch.float16 if dev == "cuda" else torch.float32   # fp16 is a GPU path only

    ck = torch.load(hf_hub_download(S2_REPO, "best.pt", token=HF_TOKEN),
                    map_location="cpu", weights_only=False)
    C = ck["cfg"]
    assert C["COND_MODE"] == "onehot", \
        "this notebook paints with the one-hot control branch; checkpoint is " + str(C["COND_MODE"])
    assert C["IMG_SIZE"] == IMG_SIZE, "the checkpoint was trained at another resolution"
    assert ck.get("control_state"), \
        "no control_state in best.pt - an untrained control branch cannot paint a layout"

    def inherit(override, key, default):
        return override if override is not None else C.get(key, default)

    P = dict(steps=int(inherit(DDIM_STEPS, "DDIM_STEPS", 30)),
             guidance=float(inherit(GUIDANCE, "GUIDANCE", 7.5)),
             cond_scale=float(inherit(COND_SCALE, "COND_SCALE", 1.0)),
             canny_scale=float(inherit(CANNY_SCALE, "CANNY_SCALE", 0.4)),
             use_canny=bool(inherit(USE_CANNY, "USE_CANNY", True)))

    u = UNet2DConditionModel.from_pretrained(C["SD_MODEL_ID"], subfolder="unet",
                                             torch_dtype=half).to(dev)
    u.requires_grad_(False)

    c = ControlNetModel.from_pretrained(C["CONTROLNET_ID"], torch_dtype=torch.float32).to(dev)
    old = c.controlnet_cond_embedding.conv_in                  # 3 -> 7 channels, BEFORE loading
    c.controlnet_cond_embedding.conv_in = nn.Conv2d(
        N_CODES, old.out_channels, kernel_size=old.kernel_size, stride=old.stride,
        padding=old.padding).to(dev, torch.float32)
    c.register_to_config(conditioning_channels=N_CODES)

    n_lora = 0
    if ck.get("lora_state"):
        u.add_adapter(LoraConfig(r=C["LORA_RANK"], lora_alpha=C["LORA_ALPHA"],
                                 lora_dropout=0.0, init_lora_weights="gaussian",
                                 target_modules=["to_q", "to_k", "to_v", "to_out.0"]))
        for p in u.parameters():
            if p.requires_grad:
                p.data = p.data.float()
        _m, _unexp = u.load_state_dict({k: v.float() for k, v in ck["lora_state"].items()},
                                       strict=False)
        assert not _unexp, "unexpected LoRA keys in the checkpoint: " + str(list(_unexp)[:4])
        n_lora = len(ck["lora_state"])

    c.load_state_dict({k: v.float() for k, v in ck["control_state"].items()})
    # control_state and lora_state are stored fp16 (Stage 2 halves them on save), so casting
    # the branch back to fp16 on a GPU recovers no precision the checkpoint ever had.
    c = c.to(half); c.requires_grad_(False)
    u.eval(); c.eval()

    v = AutoencoderKL.from_pretrained(C["SD_MODEL_ID"], subfolder="vae").to(dev).eval()
    v.requires_grad_(False)
    te = CLIPTextModel.from_pretrained(C["SD_MODEL_ID"], subfolder="text_encoder",
                                       torch_dtype=half).to(dev).eval()
    te.requires_grad_(False)
    tk = CLIPTokenizer.from_pretrained(C["SD_MODEL_ID"], subfolder="tokenizer")
    dcfg = DDIMScheduler.from_pretrained(C["SD_MODEL_ID"], subfolder="scheduler").config

    cy = None
    if P["use_canny"]:
        cy = ControlNetModel.from_pretrained(C["CANNY_ID"], torch_dtype=half).to(dev).eval()
        cy.requires_grad_(False)

    return dict(device=dev, dtype=half, unet=u, cn=c, canny=cy, vae=v, te=te, tok=tk,
                ddim=dcfg, scaling=float(v.config.scaling_factor), emb={}, cfg=C,
                params=P, n_lora=n_lora, step=ck.get("step"), arch=ck.get("arch", "?"))


def _embed(eng, caps):
    import torch
    cache = eng["emb"]
    miss = sorted({c for c in caps if c not in cache})
    if miss:
        tok = eng["tok"](miss, padding="max_length", max_length=eng["tok"].model_max_length,
                         truncation=True, return_tensors="pt")
        with torch.no_grad():
            e = eng["te"](tok.input_ids.to(eng["device"]))[0]
        for c, v in zip(miss, e):
            cache[c] = v
    return torch.stack([cache[c] for c in caps])


def _cn_unet(eng, lat, t, emb, cond, cond_scale, extra):
    import torch
    u, c = eng["unet"], eng["cn"]
    down, mid = c(lat.to(c.dtype), t, encoder_hidden_states=emb.to(c.dtype),
                  controlnet_cond=cond.to(c.dtype), conditioning_scale=cond_scale,
                  return_dict=False)
    down = list(down)
    for net_x, cond_x, sc_x in (extra or []):
        d2, m2 = net_x(lat.to(net_x.dtype), t, encoder_hidden_states=emb.to(net_x.dtype),
                       controlnet_cond=cond_x.to(net_x.dtype), conditioning_scale=sc_x,
                       return_dict=False)
        down = [a + b.to(a.dtype) for a, b in zip(down, d2)]
        mid = mid + m2.to(mid.dtype)
    down = [d.to(u.dtype) for d in down]
    return u(lat, t, encoder_hidden_states=emb,
             down_block_additional_residuals=down,
             mid_block_additional_residual=mid.to(u.dtype)).sample


def generate_images(labels, seeds, eng=None, progress=True):
    """labels: 512px class-index masks {0..5, 255}. seeds: one latent key per label.

    Returns (N, 512, 512, 3) uint8 RGB. The latent is drawn PER IMAGE on the CPU from a
    generator seeded by that image's own key - not per batch - so the output is a pure
    function of the key and does not depend on how the work was chunked.
    """
    import torch
    from diffusers import DDIMScheduler
    from contextlib import nullcontext
    eng = eng or ENG
    dev, P = eng["device"], eng["params"]
    do_cfg = P["guidance"] > 1.0
    labels = [to_512(np.asarray(l)) for l in labels]
    amp = (torch.autocast("cuda") if dev == "cuda" else nullcontext())
    out = []
    with torch.no_grad():
        cond = torch.from_numpy(np.stack([cond_from_index(l) for l in labels])).to(dev)
        emb = _embed(eng, [caption_from_index(l) for l in labels])
        neg = _embed(eng, [""] * len(labels))
        e_all = torch.cat([neg, emb]) if do_cfg else emb
        c_all = torch.cat([cond, cond]) if do_cfg else cond
        extra = []
        if P["use_canny"] and eng["canny"] is not None:
            cc = torch.from_numpy(np.stack([canny_from_index(l) for l in labels])).to(dev)
            extra = [(eng["canny"], torch.cat([cc, cc]) if do_cfg else cc, P["canny_scale"])]

        sch = DDIMScheduler.from_config(eng["ddim"]); sch.set_timesteps(P["steps"], device=dev)
        gg = torch.Generator()
        z = []
        for k in seeds:
            gg.manual_seed(int(k))
            z.append(torch.randn(4, LATENT_SIZE, LATENT_SIZE, generator=gg))
        lat = torch.stack(z).to(dev) * sch.init_noise_sigma

        t0 = time.time()
        for j, t in enumerate(sch.timesteps):
            x = torch.cat([lat] * 2) if do_cfg else lat
            x = sch.scale_model_input(x, t)
            with amp:
                eps = _cn_unet(eng, x, t, e_all, c_all, P["cond_scale"], extra)
            eps = eps.float()
            if do_cfg:
                u_, c_ = eps.chunk(2); eps = u_ + P["guidance"] * (c_ - u_)
            lat = sch.step(eps, t, lat).prev_sample
            if progress and (j + 1) % 10 == 0:
                print("    step %d/%d  (%.0f s)" % (j + 1, P["steps"], time.time() - t0),
                      flush=True)
        img = eng["vae"].decode((lat / eng["scaling"]).float()).sample
        img = (img.float() / 2 + 0.5).clamp(0, 1).cpu().numpy().transpose(0, 2, 3, 1)
    return (img * 255).round().astype(np.uint8)


if INFER != "off" and TORCH_OK:
    try:
        t0 = time.time()
        ENG = build_painter()
        P = ENG["params"]
        PAINT_STATUS = "loaded from " + S2_REPO + " (step %s)" % ENG["step"]
        print("Stage 2 ready in %.0f s on %s" % (time.time() - t0, ENG["device"]))
        print("  step %s | LoRA tensors %d | arch %s" % (ENG["step"], ENG["n_lora"], ENG["arch"]))
        print("  base %s + %s" % (ENG["cfg"]["SD_MODEL_ID"], ENG["cfg"]["CONTROLNET_ID"]))
        print("\n--- resolved sampling hyper-parameters ---")
        for nm, key, ov in (("DDIM steps", "steps", DDIM_STEPS), ("guidance", "guidance", GUIDANCE),
                            ("cond scale", "cond_scale", COND_SCALE),
                            ("canny scale", "canny_scale", CANNY_SCALE),
                            ("use canny", "use_canny", USE_CANNY)):
            print("  %-12s %-8s (%s)" % (nm, P[key],
                                         "CONTROL PANEL" if ov is not None else "inherited from best.pt"))
    except Exception as e:
        PAINT_STATUS = "not loaded - " + str(e).split("\n")[0][:200]
        print("Stage 2 NOT loaded:", PAINT_STATUS)
        print("On Kaggle this is nearly always the Internet switch: Session options ->")
        print("Internet -> On, then re-run this cell.")
else:
    PAINT_STATUS = "skipped (INFER = 'off')" if INFER == "off" else "skipped (no torch)"
    print(PAINT_STATUS)

## 6 · The frozen scorer

The third column is the same instrument Stage 3 used to measure layout fidelity: the
part-1 SegFormer-B0 block-fold-1 checkpoint, inference only, never fine-tuned here. Its
published ceiling on real fold-1 imagery is mIoU 0.4646, which is the number the
per-pair insets should be read against.

Preprocessing matches part 1 exactly — RGB, `/255`, ImageNet mean and std — and the
checkpoint is rebuilt from the `model_config` it carries rather than from a repo name,
so the keys load strictly. A checkpoint that loads with missing keys is reported as not
loaded rather than quietly producing a read-back from a half-initialised model.

In [ ]:
MODELS = {}
PRED_STATUS = {"segformer-b0": "not attempted"}


def load_b0(repo):
    import torch
    from huggingface_hub import hf_hub_download
    from transformers import SegformerForSemanticSegmentation, SegformerConfig
    ck = torch.load(hf_hub_download(repo, "best.pt", token=HF_TOKEN),
                    map_location="cpu", weights_only=False)
    cfg_d = ck.get("model_config")
    if hasattr(cfg_d, "to_dict"):
        cfg_d = cfg_d.to_dict()
    net = (SegformerForSemanticSegmentation(SegformerConfig.from_dict(cfg_d))
           if isinstance(cfg_d, dict) else
           SegformerForSemanticSegmentation.from_pretrained(repo))
    sd = ck.get("model_state", ck.get("model", ck))
    sd = {k.replace("module.", ""): v for k, v in sd.items()}
    miss, unexp = net.load_state_dict(sd, strict=False)
    # A SegFormer whose decode head did not load still returns logits - they are just
    # noise. transformers 5.x renames those keys, which is exactly how that happens.
    if miss:
        raise RuntimeError("%d keys did not load (%s ...) - check the transformers version"
                           % (len(miss), ", ".join(list(miss)[:3])))
    net.eval()
    note = " · val mIoU %.4f" % ck["metrics"]["mIoU"] if isinstance(ck.get("metrics"), dict) \
        and "mIoU" in ck.get("metrics", {}) else ""
    return net, "loaded from " + repo + note + (" (%d unexpected keys)" % len(unexp)
                                                if unexp else "")


_DEV = "cpu"
if INFER != "off" and TORCH_OK:
    try:
        MODELS["segformer-b0"], PRED_STATUS["segformer-b0"] = load_b0(HF_B0)
    except Exception as e:
        PRED_STATUS["segformer-b0"] = "not loaded - " + str(e).split("\n")[0][:170]
    try:
        import torch
        _DEV = "cuda" if torch.cuda.is_available() else "cpu"
        for _m in MODELS.values():
            _m.to(_DEV)
    except Exception:
        pass
else:
    PRED_STATUS = {k: "skipped (INFER = 'off')" for k in PRED_STATUS}

for k, v in PRED_STATUS.items():
    print("%-14s %s" % (k, v))


def predict(model, rgb, size=IMG_SIZE):
    """rgb uint8 HxWx3 -> class indices at size x size."""
    import torch
    a = np.array(Image.fromarray(rgb).resize((size, size), Image.BILINEAR), np.float32) / 255.0
    x = torch.from_numpy(((a - _MEAN) / _STD).transpose(2, 0, 1))[None].to(_DEV)
    with torch.no_grad():
        out = model(x)
        logits = out.logits if hasattr(out, "logits") else out
        if logits.shape[-1] != size:
            import torch.nn.functional as F
            logits = F.interpolate(logits, size=(size, size), mode="bilinear",
                                   align_corners=False)
    return logits.float().argmax(1)[0].cpu().numpy().astype(np.int16)


def iou_mean(pred, gt):
    """Mean IoU over the classes present in either map. Ignore pixels are dropped from
    both sides, as they are in the part-1 confusion matrix, so an unannotated region
    cannot be counted as a false positive against a class it was never labelled."""
    k = gt != 255
    p, g = pred[k], gt[k]
    vals = []
    for c in range(len(CLASSES)):
        pc, gc = (p == c), (g == c)
        u = np.logical_or(pc, gc).sum()
        if u:
            vals.append(float(np.logical_and(pc, gc).sum()) / u)
    return float(np.mean(vals)) if vals else np.nan


def scorer():
    return MODELS.get("segformer-b0")

## 7 · Painting, read-back and the figure

`prepare_rows()` does the work for one row at a time and prints what it did, so a row
that comes out wrong tells you which of the four steps it failed at: resolve the mask,
decode it, paint it, read it back.

Painted images are cached under `figs/_painted/` as `genimage{i}.png`, **BGR on disk**,
which is the Stage-3 contract — so a cached file is interchangeable with one from the
released `fake_pairs_R{r}` folder, and re-running the build cell costs nothing. Set
`REPAINT = True` to overwrite them.

The figure itself is unchanged: 190 mm `figure*`, same palette, same legend, same
caption, same `finish()` call as the six-figure notebook.

In [ ]:
def _seed_for(index, row_i):
    """Stage 3 keyed each latent to the ABSOLUTE pair index:
           torch.Generator().manual_seed(cfg.SEED + (IMAGE_SEED_BASE + 1000*R) + i)
    Reproducing that key reproduces the pair that went into Stage-4 training. Any other
    key paints a different, equally valid sample from the same generator."""
    if SEED_MODE == "stage3" and index is not None:
        return S3_SEED + IMAGE_SEED_BASE + 1000 * int(R_SET) + int(index), "stage3"
    return int(FIXED_SEED) + row_i, "fixed"


def prepare_rows():
    rows = []
    sc = scorer()
    for r, row in enumerate(FIG11_ROWS):
        print("row %d · %s" % (r, row.get("label") or ""))
        rec = dict(label=row.get("label") or "", mask=None, image=None, readback=None,
                   score=None, index=None)

        mp = find_file(row.get("mask", ""), GEN_MASKS)
        if mp is None:
            print("   no mask resolved - set FIG11_GENMASKS[%d]" % r); rows.append(rec); continue
        idx, cov = to_index(load_rgb(mp))
        if idx is None or cov < 0.2:
            print("   mask decoded only %.0f%% of its pixels - is it a mask?" % (100 * cov))
            rows.append(rec); continue
        lab = to_512(idx)
        rec["mask"] = paint(np.where(lab == 255, -1, lab).astype(np.int16))
        i = row.get("index") or _num(stem(mp))
        rec["index"] = None if i in ("", None) else int(i)

        # ---- the image: a file you named, a pair already on disk, the cache, or paint it
        rgb = None
        src = ""
        gp = find_file(row.get("image", ""), GEN_IMGS) if row.get("image") else None
        if gp is None and rec["index"] is not None:
            gp = find_file("genimage%d" % rec["index"], GEN_IMGS)
        if gp is not None:
            rgb = load_rgb(gp, bgr=True); src = "on disk: " + Path(gp).name
        if rgb is None and rec["index"] is not None and not REPAINT:
            cp = PAINTED / ("genimage%d.png" % rec["index"])
            if cp.is_file():
                rgb = load_rgb(str(cp), bgr=True); src = "cached: " + cp.name
        if rgb is None and PAINT != "off" and ENG is not None:
            seed, mode = _seed_for(rec["index"], r)
            print("   painting · seed %d (%s) · DDIM %d · guidance %.2f"
                  % (seed, mode, ENG["params"]["steps"], ENG["params"]["guidance"]))
            print("   caption: " + caption_from_index(lab))
            t0 = time.time()
            rgb = generate_images([lab], seeds=[seed])[0]
            src = "painted in %.0f s" % (time.time() - t0)
            if rec["index"] is not None:
                # written BGR, which is the Stage-3 on-disk contract, so a cached file is
                # interchangeable with one from the released fake_pairs_R{r} folder
                Image.fromarray(rgb[..., ::-1]).save(PAINTED / ("genimage%d.png" % rec["index"]))
        rec["image"] = rgb
        print("   image: " + (src or "NOT AVAILABLE (" + PAINT_STATUS + ")"))

        # ---- the read-back
        if rgb is not None and sc is not None:
            pr = predict(sc, rgb)
            rec["readback"] = paint(pr)
            rec["score"] = iou_mean(pr, lab.astype(np.int16))
            print("   read-back layout mIoU %.3f" % rec["score"])
        elif rgb is not None:
            print("   read-back skipped: " + PRED_STATUS["segformer-b0"])
        rows.append(rec)
    return rows


def build_fig11():
    rows = prepare_rows()
    cols = ["generated mask", "generated image", "frozen scorer read-back"]
    fig, axs = plt.subplots(3, 3, figsize=(W2, W2 * 0.80), layout="constrained")
    for r in range(3):
        rec = rows[r] if r < len(rows) else {}
        notes = ["set FIG11_GENMASKS[%d]" % r, "generated image not found",
                 "scorer read-back\nnot available"]
        for c, arr in enumerate([rec.get("mask"), rec.get("image"), rec.get("readback")]):
            show(axs[r, c], arr, notes[c])
            if r == 0:
                axs[r, c].set_title(cols[c], fontsize=7, pad=2)
        if rec.get("score") is not None and rec.get("readback") is not None:
            axs[r, 2].text(0.97, 0.04, "layout mIoU %.3f" % rec["score"],
                           transform=axs[r, 2].transAxes, ha="right", va="bottom",
                           fontsize=6.0, color="white",
                           bbox=dict(fc="0.15", ec="none", pad=1.2, alpha=0.75))
        axs[r, 0].set_ylabel(rec.get("label") or "", fontsize=6.5)

    class_legend(fig)
    finish(fig, "fig11_synthetic_pairs", "Fig. 11", "Section 6.4",
           "at the end of the subsection, after the presence-rate sentence", "figure*",
           "Three synthetic pairs from fake pairs R" + str(R_SET) + " spanning the fidelity "
           "range: the mask drawn by the Stage-1 diffusion model, the image Stage 2 paints from "
           "it, and the frozen scorer's read-back of that image against the mask that generated "
           "it, with the per-pair layout intersection over union inset. The bottom row shows the "
           "failure that drives the downstream result: river arriving as a wide fragmented blob "
           "rather than a narrow continuous ribbon.")
    plt.show()
    return rows

---
## 8 · BUILD

In [ ]:
ROWS = build_fig11()

## 9 · Register and download

Writes `figs/figure_register_fig11.csv` — its own file, so it cannot overwrite the
register the other twenty-one figures wrote — and zips `figs/` for a one-click
download. The PDF is the submission file; the PNG is a 600 dpi preview.

In [ ]:
import csv

with open(OUT / "figure_register_fig11.csv", "w", newline="") as fh:
    w = csv.DictWriter(fh, fieldnames=["file", "figure", "section", "insertion_point",
                                       "float_env", "size", "caption"])
    w.writeheader()
    for row in REGISTER:
        w.writerow(row)

for r in (ROWS if "ROWS" in dir() else []):
    if r.get("index") is not None and r.get("score") is not None:
        print("pair %-5s layout mIoU %.4f   %s" % (r["index"], r["score"], r["label"]))

zipname = shutil.make_archive("geonusaf_fig11", "zip", root_dir=str(OUT))
print("\n" + str(OUT / "fig11_synthetic_pairs.pdf"))
print(zipname)
try:
    from IPython.display import FileLink, display
    display(FileLink(Path(zipname).name))
except Exception:
    pass

---
## Notes

- **The three columns must agree on the same mask.** The condition fed to the painter
  and the target the scorer is measured against are the same `lab` array, built once
  per row from the file you named. There is no second decoding path that could disagree
  with the one drawn in column 1.
- **Colour masks are fine.** `to_index` measures the file against four candidate lookup
  tables — released palette, display palette, and both byte-reversed — and keeps the one
  that explains the most pixels. Your extracted colour masks decode through the released
  table; the `{0..5} + 255` contract masks take the index-coded branch instead.
- **Cost.** First run downloads SD 1.5 (~4 GB), the two ControlNets and the Stage-2
  checkpoint (729 MB). Painting is about 9 s per image on a T4 at DDIM 30, so three rows
  is well under a minute once everything is resident; on CPU expect several minutes each.
- **Repainting is not bit-identical to the released file.** The latent is reproduced
  exactly, but cuDNN picks algorithms by batch size and device, so a pair painted here
  one at a time can differ from the released `genimage{i}.png` in the last bits. It is
  the same sample, not a different draw — visually indistinguishable, numerically not
  equal. If the caption must claim the exact artefact used in Stage-4 training, point
  `FIG11_GENIMAGES` at the released files instead and the notebook will use them and
  only compute the read-back.
- **Choosing the three rows.** The labels assume a high, a middling and a river-bearing
  low-fidelity pair. The printed per-row layout mIoU tells you whether the masks you
  picked actually span that range; swap indices until they do. Stage 3 measured the
  whole R1 pool at 0.4395 against a real-image ceiling of 0.4646.
- **If the scorer fails to load**, the middle column still paints and the third column
  states what is missing rather than showing something invented. The usual cause is
  `transformers` 5.x, which renames the decode-head keys.
